# Programming a micro:bit

A **micro:bit** is a pocket-sized computer with 25 lights (LEDs), two buttons, a speaker and sensors for movement, light and temperature. You can find out more at https://microbit.org.

In Codetto you program it in two ways at once:

- **micro:bit cells** run **MicroPython** on the micro:bit itself. MicroPython is a small version of Python made for devices like this, and the `microbit` library gives it control of the lights, buttons and sensors.
- **Code cells** run Python in your browser, as usual. With `codetto.devices` they can read what the micro:bit sends back, then plot it, save it or use it in a game.

**You'll need** a micro:bit, a USB cable and Chrome or Edge. Sound needs a micro:bit V2, which has a built-in speaker (the V1 has none).

# Connecting and deploying

A micro:bit cell looks like a code cell, but it has its own buttons:

1. Plug your micro:bit into your computer with the USB cable.
2. Click **Connect** in the cell below and choose your micro:bit in the list your browser shows.
3. Click **Deploy**. This copies the program onto the micro:bit, and it starts straight away.

Every micro:bit cell in this notebook shares the same connection, so you only need to connect once. Each Deploy replaces the program that was on the micro:bit before.

`from microbit import *` brings in everything from the `microbit` library. `display.scroll()` slides text across the lights, and `display.show()` shows a picture. The program ends there, so the happy face stays on.

# The LED display

The display is a 5 × 5 grid of lights. As well as built-in pictures such as `Image.HEART`, `Image.YES` and `Image.ARROW_N`, you can control each light yourself with `display.set_pixel(x, y, brightness)`:

- `x` is the column and `y` is the row, both from 0 to 4, starting at the top left
- `brightness` goes from 0 (off) to 9 (brightest)

`sleep(500)` pauses the program for 500 milliseconds, which is half a second. Deploy this program and watch the display.

# Buttons and loops

A program that keeps reacting to the buttons runs inside a `while True:` loop, which never ends. Inside the loop:

- `button_a.was_pressed()` is `True` once for each press since the last time you asked
- `button_a.is_pressed()` is `True` for as long as the button is held down

`print()` on the micro:bit sends a line of text back to your computer over the USB cable. The cell's **Serial Monitor** tab shows those lines.

Deploy this counter, press **A** a few times and **B** to reset it, then open the Serial Monitor.

# Sensors

The `microbit` library reads the micro:bit's sensors:

- `accelerometer.get_x()`, `get_y()` and `get_z()`: how the micro:bit is tilted or moving, in milli-g. Gravity alone gives about 1000.
- `display.read_light_level()`: how bright it is around the micro:bit, from 0 (dark) to 255 (bright). The LEDs themselves measure it.
- `temperature()`: the temperature in °C, measured inside the micro:bit, so usually a little above room temperature.

This program sends a reading 10 times a second. Each reading is one line of numbers separated by commas, which is easy for a program on your computer to split apart. Deploy it, then tilt the micro:bit and cover it with your hand while you watch the Serial Monitor.

# Reading the micro:bit from Python

Now switch to Python in your browser. Keep the sensor program running on the micro:bit.

`devices.MicroBit.connect()` attaches to the connection the micro:bit cell opened. It doesn't open one itself, so click **Connect** above first. Then:

- `bit.read_line()` waits for the next line the micro:bit prints and returns it as text, or `None` if the micro:bit was unplugged
- `bit.clear_buffer()` throws away lines that arrived earlier, so you start with fresh readings

`parse_reading()` turns a line such as `"-120,980,34"` into a list of three numbers. It returns `None` for anything else, such as half a line that was arriving when the cell started.

In [ ]:
from codetto import devices

def parse_reading(line):
  parts = line.strip().split(",")
  if len(parts) != 3:
    return None
  try:
    return [int(part) for part in parts]
  except ValueError:
    return None

bit = devices.MicroBit.connect()
bit.clear_buffer()

for i in range(10):
  line = bit.read_line()
  if line is None:
    print("The micro:bit was disconnected")
    break
  reading = parse_reading(line)
  if reading is not None:
    x, y, light = reading
    print(f"x={x:5}   y={y:5}   light={light:3}")

# Recording and plotting

Once the readings are in Python, you can use any library you like with them. This cell records for a few seconds while you tilt the micro:bit, then draws the readings with matplotlib. Choose how long to record, run the cell, and start moving.

In [ ]:
#@title Record and plot
#@alt Two line charts over time: the micro:bit's tilt (x and y, in milli-g) and its light level
SECONDS = 5 #@param {type:"slider", min:2, max:20, step:1}

import matplotlib.pyplot as plt

readings = []
bit.clear_buffer()
print(f"Recording for {SECONDS} seconds: tilt your micro:bit!")

while len(readings) < SECONDS * 10:  # the micro:bit sends 10 readings a second
  line = bit.read_line()
  if line is None:
    print("The micro:bit was disconnected")
    break
  reading = parse_reading(line)
  if reading is not None:
    readings.append(reading)

print(f"Recorded {len(readings)} readings")

times = [i / 10 for i in range(len(readings))]
fig, (tilt_chart, light_chart) = plt.subplots(2, 1, sharex=True, figsize=(8, 5))

tilt_chart.plot(times, [r[0] for r in readings], label="x (left and right)")
tilt_chart.plot(times, [r[1] for r in readings], label="y (forward and back)")
tilt_chart.set_ylabel("milli-g")
tilt_chart.legend()

light_chart.plot(times, [r[2] for r in readings], color="orange")
light_chart.set_ylabel("light level")
light_chart.set_xlabel("seconds")

plt.show()

# Things to Try

- Add `temperature()` to the sensor program as a fourth number, then update `parse_reading()` and add it to the chart.
- Save your readings to a file such as `/attachments/readings.csv`, so they stay with this notebook.
- Make a step counter: `accelerometer.was_gesture("shake")` is `True` once for each shake. Show the count on the display and send it to your notebook.
- Turn the micro:bit into a game controller: print `"A"` or `"B"` when a button is pressed, and make a code cell react to it, for example by playing a sound with `codetto.audio`.

# Check Your Understanding